## Classic Environment Preflight

This notebook requires the classic runtime. If this check fails, rebuild with `CLASSIC=1 make notebooks-build`, restart the container, and select kernel **Python 3 (classic-langchain)**.


In [ ]:
import os
import sys

def _classic_fail(reason: str) -> None:
    raise RuntimeError(
        f"Classic runtime preflight failed: {reason}\n"
        "Fix:\n"
        "1. CLASSIC=1 make notebooks-build\n"
        "2. make notebooks-up\n"
        "3. In Jupyter, select kernel: Python 3 (classic-langchain)"
    )

kernel_name = os.getenv("JPY_KERNEL_NAME", "")
prefix = sys.prefix.lower()
if "venv-classic" not in prefix and "classic" not in kernel_name.lower():
    _classic_fail(f"detected sys.prefix={sys.prefix!r}, JPY_KERNEL_NAME={kernel_name!r}")

try:
    import langchain  # noqa: F401
except Exception as exc:
    _classic_fail(f"langchain import failed: {exc}")

print("Classic preflight passed.")


# Classic Notebook
> Classic track note: This notebook demonstrates legacy/classic LangChain-era patterns for evaluation and comparison.
> Prefer the modern equivalents in `lessons/2026/langchain/` for current APIs and recommended techniques.


In [ ]:
%pip -q install langchain langchain-openai

In [ ]:
%pip install python-dotenv
import os
from dotenv import load_dotenv
load_dotenv()

ENABLE_LANGSMITH = bool(os.getenv("LANGCHAIN_API_KEY"))
print(f"LANGCHAIN_API_KEY available: {ENABLE_LANGSMITH}")


In [ ]:
if not ENABLE_LANGSMITH:
    print("Skipping LangSmith dataset setup (LANGCHAIN_API_KEY not set)")
else:
    from langsmith import Client as LangSmithClient
    
    DATASET_NAME='Config mgmt Rap Battle Dataset'
    client = LangSmithClient()
    datasets=client.list_datasets()
    for dataset in datasets:
        if (dataset.name) == DATASET_NAME:
            client.delete_dataset(dataset_name=DATASET_NAME)
            print(dataset)

In [ ]:

# Inputs are provided to your model, so it know what to generate
dataset_inputs = [
    "a rap battle between Chef and Puppet",
    "a rap battle between Ansible and Pulumi",
    # ... add more as desired
]


In [ ]:


# Storing inputs in a dataset lets us
# run chains and LLMs over a shared set of examples.
dataset = client.create_dataset(
    dataset_name=DATASET_NAME,
    description="Rap battle prompts.",
)
client.create_examples(
    inputs=[{"question": q} for q in dataset_inputs],
#    outputs=dataset_outputs,
    dataset_id=dataset.id,
)


In [ ]:
from langchain import prompts
from langchain.schema import output_parser

# Define your runnable or chain below.
prompt = prompts.ChatPromptTemplate.from_messages(
    [("system", "You are a helpful AI assistant."), ("human", "{question}")]
)

from langchain_openai import ChatOpenAI
chat = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

chain = prompt | chat | output_parser.StrOutputParser()
answer=chain.invoke({"question":"A rap battle between Kris and Toshaan"})
print(answer)

In [ ]:
if not ENABLE_LANGSMITH:
    print("Skipping LangSmith evaluation run (LANGCHAIN_API_KEY not set)")
else:
    from langsmith.evaluation import EvaluationResult, run_evaluator
    from langsmith.schemas import Example, Run

    @run_evaluator
    def is_empty(run: Run, example: Example | None = None):
        model_outputs = run.outputs.get("output", "")
        score = not str(model_outputs).strip()
        return EvaluationResult(key="is_empty", score=score)

    chain_results = client.run_on_dataset(
        dataset_name=DATASET_NAME,
        llm_or_chain_factory=chain,
        evaluation={"custom_evaluators": [is_empty]},
        concurrency_level=5,
        verbose=True,
    )


In [ ]:
from pprint import pprint 

pprint(chain_results)